# 04 · Demo: chat with FairRent Akure

Pipeline: **message → Gemma extracts fees → fee engine checks them against survey data → Gemma writes the reply → safeguard**.

The safeguard checks every number in the model's reply against the engine result. If any number is not in the result,
the demo shows the template reply instead, so tenants never see an invented figure.

**Runs on:** Colab T4 GPU. Needs the LoRA adapter from notebook 02.

In [2]:
# --- Setup: get the project code (works in Colab and locally) ---
REPO_URL = "https://github.com/olaideagbolade/fairrent-akure.git"  # <-- change to your repo

import os, sys
if os.path.exists("../src/fairrent"):          # running locally from notebooks/
    os.chdir("..")
elif not os.path.exists("src/fairrent"):       # running in Colab
    folder = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
    if not os.path.exists(folder):
        os.system(f"git clone -q {REPO_URL}")
    os.chdir(folder)
sys.path.insert(0, "src")
print("Working in:", os.getcwd())

Working in: /content/fairrent-akure


In [3]:
!pip uninstall -y -q torchao
!pip install -q "transformers>=4.50" "peft>=0.13" accelerate gradio

In [4]:
import pandas as pd
from huggingface_hub import login
try:
    from google.colab import userdata
    login(userdata.get("HF_TOKEN"))
except Exception as e:
    print("Using existing Hugging Face login.", e)

from fairrent.engine import FeeEngine
from fairrent.llm import DEFAULT_MODEL_ID, FairRentAssistant, load_model

ADAPTER_PATH = "fairrent-gemma3-1b-lora"
if not os.path.exists(ADAPTER_PATH):
    from google.colab import drive
    drive.mount("/content/drive")
    ADAPTER_PATH = "/content/drive/MyDrive/fairrent/fairrent-gemma3-1b-lora"

model, tokenizer = load_model(DEFAULT_MODEL_ID, ADAPTER_PATH)
engine = FeeEngine(pd.read_csv("data/tenants_clean.csv", keep_default_na=False, na_values=[""]))
assistant = FairRentAssistant(model, tokenizer, engine)

Mounted at /content/drive


config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.00GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [5]:
for msg in [
    "Agent wan collect money for selfcon for southgate. Rent na 280k, agency 56k, caution 50k, development fee 40k. E make sense?",
    "How much is a single room around North Gate?",
    "Ẹ n lẹ́. Mo fẹ́ gba self-contain ní Apatapiti. Rent jẹ́ 200k, agency fee jẹ́ 20k. Ṣé ó dára?",
    "Who won the match yesterday?",
]:
    out = assistant.chat(msg)
    print("USER:", msg)
    print("FAIRRENT:", out["reply"])
    print(f"(model reply grounded: {out['model_reply_grounded']}, fallback used: {out['used_fallback']})")
    print("-" * 80)

USER: Agent wan collect money for selfcon for southgate. Rent na 280k, agency 56k, caution 50k, development fee 40k. E make sense?
FAIRRENT: See how your money take be (Self-contain, FUTA South Gate):

- Rent ₦280,000: ⚠️ E pass normal
- Agency fee ₦56,000: ⚠️ E pass normal (20% of rent; normal na 10% or less)
- Caution fee ₦50,000: ⚠️ E pass normal (18% of rent; normal na 15% or less)
- Development fee ₦40,000: 🚩 Red flag (no be many people dey pay this one)

Most people wey report Self-contain in FUTA South Gate dey pay ₦172,500–₦250,000 for one year (16 reports).
Total wey dem ask you to pay: ₦426,000.
Check these ones well: Rent, Agency fee, Caution fee, Development fee.

Wetin you fit do:
1. Collect receipt for every money wey you pay.
2. Confirm the rent with the landlord yourself before you pay.
3. Beg the agent make e reduce the fees wey pass normal.
4. Ask wetin development fee dey cover and who dey collect am.

Na from Akure tenant survey we get these numbers. Na advice, no b

## Interactive demo (share the public link for user testing)

In [6]:
import gradio as gr

def respond(message, history):
    return assistant.chat(message)["reply"]

gr.ChatInterface(
    respond, title="FairRent Akure",
    description="Paste your rent and fee breakdown (English, Pidgin or Yoruba). Guidance from Akure tenant survey data, not legal advice.",
    examples=["Selfcon for South Gate. Rent 250k, agency 50k, agreement 25k, caution 30k. Is this fair?",
              "How much is self-contain for northgate?"],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ec8ac1d78ebace3147.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
